# clean text from emails, html tags, etc.

In [17]:
import re
import pandas as pd
from tqdm import tqdm

In [2]:
def cleanhtml(raw_html):
  cleantext = re.sub('<.*?>', '', raw_html)
  cleantext = re.sub('(http\S+)', '', cleantext)
  cleantext = re.sub('([\w\.\-\_]+@[\w\.\-\_]+)', '', cleantext)
  return cleantext

In [ ]:
# clean text and lede
data = pd.read_csv('sta_full_data.csv')

data['cleaned_text'] = data.apply(
    lambda row: cleanhtml(row['text']) if row['text'] else row['text'],
    axis=1
)
data['cleaned_lede'] = data.apply(
    lambda row: cleanhtml(row['lede']) if row['lede'] else row['lede'],
    axis=1
)

In [12]:
data['cleaned_text'] = data.apply(
    lambda row: cleanhtml(row['text']) if row['text'] else row['text'],
    axis=1
)
data['cleaned_lede'] = data.apply(
    lambda row: cleanhtml(row['lede']) if row['lede'] else row['lede'],
    axis=1
)

# split text into sentences and mark meaningful and not meaningful sentences

In [ ]:
import spacy
import shortuuid
from sentence_splitter import SentenceSplitter
from tqdm import tqdm

# Load NLP model and sentence splitter once
nlp = spacy.load('en_core_web_trf')
splitter = SentenceSplitter(language='en')

# define pos tags
allowed_pos = {'ADJ', 'NOUN', 'VERB', 'PRON', 'PROPN', 'AUX'}

def process_splits(art_id, date, splits, is_lede):
    result = {
        'id': art_id,
        'date': date,
        'article': []
    }

    for i, sentence in enumerate(splits):
        sentence = str(sentence).strip()
        if not sentence:
            continue

        doc = nlp(sentence)
        pos_tags = [token.pos_ for token in doc if token.pos_ in allowed_pos]

        entry = {
            'sen_id': shortuuid.uuid(),
            'seq_nr': i,
            'text': sentence,
            'status': 'true_sentence' if len(pos_tags) > 7 and 'VERB' in pos_tags else 'false_sentence'
        }

        if entry['status'] == 'false_sentence':
            entry['is_lede'] = is_lede

        result['article'].append(entry)

    return result

# Process all rows
results = []
for _, row in tqdm(data.iterrows(), total=len(data)):
    art_id, date = row['id'], row['date']
    
    for text, is_lede in [(row.cleaned_text, False), (row.cleaned_lede, True)]:
        splits = splitter.split(text)
        results.append(process_splits(art_id, date, splits, is_lede))